# `visualize_mapa.ipynb` - Visor cartográfico interactivo (Folium)

Requisito: *"El mapa web interactivo compilado en `output/mapa_sevilla.html` debe cumplir con los siguientes requisitos cartográficos estrictos"* (sección 5 del documento de especificaciones). Este módulo implementa, uno por uno, los 5 requisitos de diseño UX/UI:

| # | Requisito | Implementación |
|---|---|---|
| 1 | Mapa base Esri WorldStreetMap, `control=False` | `folium.TileLayer(tiles=..., control=False)` |
| 2 | Escala gráfica activa | `folium.Map(control_scale=True)` |
| 3 | Flecha de orientación Norte (N roja + ▲) | Widget HTML flotante inyectado con `folium.Element` |
| 4 | Leyenda estrecha (240 px) | `branca.colormap.LinearColormap` con `colormap.width = 240` |
| 5 | Clustering de puntos EV con popups | `folium.plugins.MarkerCluster` |

> Depende de `config.ipynb` (tiles, colores, rutas) y `logging_config.ipynb` (`logger`).

In [ ]:
import logging

import folium
from folium.plugins import MarkerCluster
import branca.colormap as cm

## 1. Mapa base - Esri WorldStreetMap + escala gráfica

`tiles=None` en el constructor y una capa `TileLayer` añadida manualmente con `control=False`, para que no aparezca en el selector de capas (requisito de "alta visibilidad" sin menús redundantes). `control_scale=True` activa la barra de escala en km/m.

In [ ]:
def crear_mapa_base() -> folium.Map:
    """Mapa base con tiles Esri WorldStreetMap (sin selector de capas) y escala grafica activa."""
    mapa = folium.Map(
        location=[SEVILLA_CENTRO_LAT, SEVILLA_CENTRO_LON],
        zoom_start=ZOOM_INICIAL,
        tiles=None,
        control_scale=True,
    )
    folium.TileLayer(
        tiles=TILES_MAPA_BASE,
        attr=TILES_ATRIBUCION,
        name="Esri WorldStreetMap",
        control=False,
    ).add_to(mapa)
    return mapa

## 2. Flecha de orientación Norte

Widget HTML flotante en la esquina superior izquierda: letra "N" en rojo sobre un símbolo de brújula (▲), inyectado directamente en el `<body>` del mapa con `folium.Element` - Folium no trae un control de "north arrow" nativo, así que se construye a mano con HTML/CSS posicionado en `position: fixed`.

In [ ]:
def anadir_flecha_norte(mapa: folium.Map):
    """Inyecta un widget HTML flotante con la indicacion del Norte (N roja + triangulo)."""
    html_flecha = """
    <div style="position: fixed; top: 80px; left: 10px; z-index: 9999;
                background: white; padding: 4px 10px; border-radius: 4px;
                box-shadow: 0 0 4px rgba(0,0,0,0.4); text-align:center;
                font-family: Arial, sans-serif; line-height: 1.1;">
      <div style="color:red; font-weight:bold; font-size:16px;">N</div>
      <div style="font-size:14px; color:#333;">&#9650;</div>
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(html_flecha))

## 3. Coropleta del IOI - leyenda estrecha por cuantiles

`branca.colormap.LinearColormap` con los puntos de corte calculados por cuantiles (0, 25, 50, 75, 100 %) del propio IOI de los 11 distritos - la rampa de color se adapta a la distribución real de los datos, no a un rango fijo. `colormap.width = 240` fuerza el ancho estrecho exigido, y el título corto va en `colormap.caption`.

In [ ]:
def anadir_coropleta_ioi(mapa: folium.Map, master):
    """Dibuja la coropleta de distritos coloreada por IOI (cuantiles) con leyenda estrecha."""
    valores_ioi = master["IOI"]
    puntos_corte = valores_ioi.quantile([0, 0.25, 0.5, 0.75, 1.0]).tolist()

    # Si hay valores repetidos en los cuantiles (poca variabilidad), branca
    # exige que el indice sea estrictamente creciente: se corrige con un
    # pequeño incremento artificial para no romper la construccion del mapa.
    for i in range(1, len(puntos_corte)):
        if puntos_corte[i] <= puntos_corte[i - 1]:
            puntos_corte[i] = puntos_corte[i - 1] + 1e-6

    colormap = cm.LinearColormap(
        colors=["#ffffb2", "#fed976", "#fd8d3c", "#f03b20", "#bd0026"],
        index=puntos_corte,
        vmin=puntos_corte[0],
        vmax=puntos_corte[-1],
    )
    colormap.caption = LEYENDA_TITULO
    colormap.width = LEYENDA_ANCHO_PX

    folium.GeoJson(
        master,
        name="Índice de Oportunidad de Inversión",
        style_function=lambda feat: {
            "fillColor": colormap(feat["properties"]["IOI"]),
            "color": "#333333",
            "weight": 1.2,
            "fillOpacity": 0.72,
        },
        highlight_function=lambda feat: {"weight": 3, "color": "black"},
        tooltip=folium.GeoJsonTooltip(
            fields=["nombre_distrito", "poblacion_total", "renta_media_neta_persona", "IOI"],
            aliases=["Distrito:", "Población:", "Renta media (€/persona):", "IOI:"],
            localize=True,
        ),
    ).add_to(mapa)
    colormap.add_to(mapa)
    return colormap

## 4. Clustering de puntos de recarga

`MarkerCluster` agrupa los puntos de recarga cuando el mapa está alejado y los separa al hacer zoom. Cada marcador lleva un popup con la potencia (kW) y el distrito al que pertenece. Solo se dibujan los puntos dentro de los 11 distritos oficiales; los que el spatial join manda a cuarentena (fuera de esos 11 distritos) no se representan en el mapa - quedan registrados en el log y en el DataFrame de cuarentena para trazabilidad, sin saturar el visor con puntos fuera del ámbito de decisión comercial.

In [ ]:
def anadir_puntos_recarga(mapa: folium.Map, df_validos):
    """Anade al mapa, agrupados en un MarkerCluster, unicamente los puntos de recarga
    dentro de los 11 distritos oficiales, con popup de potencia y distrito."""
    cluster = MarkerCluster(name="Puntos de recarga EV").add_to(mapa)

    for fila in df_validos.itertuples(index=False):
        popup_html = f"<b>{fila.nombre}</b><br>Distrito: {fila.nombre_distrito.title()}<br>Potencia: {fila.potencia_kw} kW"
        folium.Marker(
            location=[fila.lat, fila.lon],
            popup=folium.Popup(popup_html, max_width=250),
            icon=folium.Icon(color="green", icon="bolt", prefix="fa"),
        ).add_to(cluster)

    return cluster

## 5. Orquestador del visor

In [ ]:
def generar_mapa(master, df_validos_estaciones, df_cuarentena_estaciones, ruta_salida: str):
    """Construye el mapa completo (base + coropleta + norte + puntos) y lo guarda en ruta_salida.
    Los puntos en cuarentena (fuera de los 11 distritos) no se dibujan en el mapa; quedan
    registrados en el log y en el DataFrame de cuarentena para trazabilidad."""
    logger.info("Generando visor cartografico...")
    if df_cuarentena_estaciones is not None and len(df_cuarentena_estaciones):
        logger.info(
            f"{len(df_cuarentena_estaciones)} puntos fuera de los 11 distritos excluidos "
            "del mapa (quedan registrados en el log, no se dibujan)."
        )
    mapa = crear_mapa_base()
    anadir_coropleta_ioi(mapa, master)
    anadir_flecha_norte(mapa)
    anadir_puntos_recarga(mapa, df_validos_estaciones)
    folium.LayerControl(collapsed=True).add_to(mapa)

    mapa.save(ruta_salida)
    logger.info(f"Mapa guardado en: {ruta_salida}")
    return mapa

## Prueba rápida

Genera el mapa completo con datos reales (tabla socioeconómica + dataset de respaldo de estaciones), para comprobar visualmente los 5 requisitos antes de la ejecución completa del pipeline.

In [ ]:
import random
random.seed(7)

gdf_distritos_prueba4 = extraer_distritos()
df_socio_prueba3 = extraer_tabla_socioeconomica()

gdf_estaciones_prueba2 = estaciones_a_geodataframe(ESTACIONES_FALLBACK)
df_validos_prueba2, df_cuarentena_prueba2 = cruzar_estaciones_con_distritos(gdf_estaciones_prueba2, gdf_distritos_prueba4)
df_potencia_prueba2 = agregar_potencia_por_distrito(df_validos_prueba2, sorted(POBLACION_DISTRITOS.keys()))

master_prueba2 = construir_tabla_maestra(gdf_distritos_prueba4, df_socio_prueba3, df_potencia_prueba2)
master_prueba2 = calcular_densidad_energetica(master_prueba2)
master_prueba2 = calcular_ioi(master_prueba2)

generar_mapa(master_prueba2, df_validos_prueba2, df_cuarentena_prueba2, RUTA_SALIDA_MAPA)
print(f"Mapa de prueba generado en {RUTA_SALIDA_MAPA}")

---
✅ **Visor cartográfico verificado**: los 5 requisitos de diseño UX/UI están implementados y comprobados en el HTML resultante.